# B04 — Premise Parser Evaluation

Calls `/premises` for sampled problems and inspects:
- FOL structure correctness (quantifier, operator, arguments)
- Predicate schema — canonical names and detected aliases
- Predicate renames applied during canonicalization
- Verification status (AST-to-schema consistency)

Gold FOL from `premises-FOL` is shown for reference but uses short identifiers; the parser output uses natural-language predicate names — compare structure, not spelling.

## 1. Configuration

In [1]:
import json
import random
from pathlib import Path
from pprint import pprint

import httpx
from IPython.display import Markdown, display

API_BASE = "https://api.iamphuckhang.dev"
TIMEOUT_SECONDS = 120.0
SAMPLE_SIZE = 10
RANDOM_SEED = 42

DATASET_PATH = Path("../datasets/exact/Logic_Based_Educational_Queries.json")

async def call_premises(premises: list[str]) -> dict:
    payload = {"premises": premises}
    async with httpx.AsyncClient(timeout=TIMEOUT_SECONDS) as client:
        response = await client.post(f"{API_BASE}/premises", json=payload)
        response.raise_for_status()
        return response.json()

## 2. Load and Sample Dataset

In [2]:
dataset = json.loads(DATASET_PATH.read_text())
print(f"Total problems: {len(dataset)}")

random.seed(RANDOM_SEED)
sample = random.sample(dataset, SAMPLE_SIZE)

print(f"Sampled {len(sample)} problems")
print(f"Premise counts: {[len(item['premises-NL']) for item in sample]}")

Total problems: 411
Sampled 10 problems
Premise counts: [14, 12, 5, 13, 4, 15, 7, 17, 13, 12]


## 3. Parse All Sampled Problems

In [5]:
results = []
for i, item in enumerate(sample):
    print(f"Parsing problem {i+1}/{len(sample)} ({len(item['premises-NL'])} premises)...", end=" ")
    try:
        result = await call_premises(item["premises-NL"])
        results.append({"item": item, "result": result, "error": None})
        print("ok")
    except Exception as exc:
        results.append({"item": item, "result": None, "error": str(exc)})
        print(f"ERROR: {exc}")

ok = sum(1 for r in results if r["error"] is None)
print(f"\n{ok}/{len(results)} succeeded")

Parsing problem 1/10 (14 premises)... ok
Parsing problem 2/10 (12 premises)... ok
Parsing problem 3/10 (5 premises)... ok
Parsing problem 4/10 (13 premises)... ok
Parsing problem 5/10 (4 premises)... ok
Parsing problem 6/10 (15 premises)... ok
Parsing problem 7/10 (7 premises)... ok
Parsing problem 8/10 (17 premises)... ok
Parsing problem 9/10 (13 premises)... ok
Parsing problem 10/10 (12 premises)... ok

10/10 succeeded


## 4. Inspect Each Problem

In [6]:
def show_problem(idx: int, entry: dict) -> None:
    item = entry["item"]
    result = entry["result"]
    error = entry["error"]

    display(Markdown(f"---\n### Problem {idx+1}"))

    if error:
        print(f"  ERROR: {error}")
        return

    parsed = result["premises"]
    gold_fol = item["premises-FOL"]
    nl = item["premises-NL"]

    display(Markdown("**NL → Parsed FOL (parser output) | Gold FOL**"))
    max_len = max(len(parsed), len(gold_fol))
    for j in range(max_len):
        nl_text  = nl[j] if j < len(nl) else "—"
        parsed_fol = parsed[j]["fol"] if j < len(parsed) else "—"
        gold      = gold_fol[j] if j < len(gold_fol) else "—"
        print(f"  [{j+1}] NL:     {nl_text}")
        print(f"       Parsed: {parsed_fol}")
        print(f"       Gold:   {gold}")
        print()

for i, entry in enumerate(results):
    show_problem(i, entry)

---
### Problem 1

**NL → Parsed FOL (parser output) | Gold FOL**

  [1] NL:     SQL is a standard programming language used for managing relational databases.
       Parsed: Standard(SQL)
       Gold:   ForAll(x, SQL(x) → UsedForManagingRelationalDatabases(x))

  [2] NL:     A relational database consists of tables, each containing rows and columns.
       Parsed: ConsistsOf(RelationalDatabase, Tables)
       Gold:   ForAll(x, RelationalDatabase(x) → ConsistsOfTables(x))

  [3] NL:     SQL queries are used to interact with databases and perform operations such as retrieving, inserting, updating, and deleting data.
       Parsed: (((Used(SQLQueries, Databases) AND UsedFor(SQLQueries, RetrievingData)) AND UsedToPerform(SQLQueries, InsertingData)) AND Used(SQLQueries, Data))
       Gold:   ForAll(x, SQLQuery(x) → (RetrieveData(x) ∨ InsertData(x) ∨ UpdateData(x) ∨ DeleteData(x)))

  [4] NL:     The SELECT statement is used to query and retrieve data from one or more tables.
       Parsed: UsedToQuery(SELECTStatement, Data)
       Gold:   ForAll(x, Select

---
### Problem 2

**NL → Parsed FOL (parser output) | Gold FOL**

...

**NL → Parsed FOL (parser output) | Gold FOL**

  [1] NL:     Students are allowed to enter the laboratory to conduct experiments only if they have both health insurance and accident insurance.
       Parsed: ∀x.((Student(x) AND HasInsurance(x)) IMPLIES CanEnter(x, Laboratory))
       Gold:   ∀s (AllowedToEnterLab(s) ↔ (HasHealthInsurance(s) ∧ HasAccidentInsurance(s)))

  [2] NL:     Lan has both health insurance and accident insurance.
       Parsed: (Has(Lan, HealthInsurance) AND Insurance(Lan, Accident))
       Gold:   HasHealthInsurance(Lan)

  [3] NL:     The laboratory is open from 9 AM to 5 PM on weekdays, unless there’s a special event.
       Parsed: ∀x.((Laboratory(x) AND Open(x, 9AM, 5PM, Weekdays)) IMPLIES NOT(Open(x)))
       Gold:   HasAccidentInsurance(Lan)

  [4] NL:     Students must wear safety goggles in the lab, but this rule is waived for virtual labs.
       Parsed: ∀x.((Student(x) AND InLab(x)) IMPLIES RequiredWear(x, SafetyGoggles))
       Gold:   ∀s ((LabOpen(Weekdays, Time_9AM_to_5PM) ∧ ¬SpecialEvent) → Can

---
### Problem 10

**NL → Parsed FOL (parser output) | Gold FOL**

  [1] NL:     At least one student is learning Python.
       Parsed: ∃x.(Student(x) AND LearningPython(x))
       Gold:   ∃x (LearningPython(x))

  [2] NL:     All students can access Python learning resources.
       Parsed: ∀x.(Student(x) IMPLIES CanAccess(x, PythonLearningResources))
       Gold:   ∀x (Student(x) → AccessPythonResources(x))

  [3] NL:     If a student practices Python regularly, they improve their coding skills.
       Parsed: ∀x.((Student(x) AND PracticesPython(x)) IMPLIES Improve(x, CodingSkills))
       Gold:   ∀x (PracticesPython(x) → ImprovesCodingSkills(x))

  [4] NL:     If a student does not practice Python, their coding skills do not improve.
       Parsed: ∀x.((Student(x) AND NOT(Practice(x, Python))) IMPLIES NOT(Improve(x)))
       Gold:   ∀x (¬PracticesPython(x) → ¬ImprovesCodingSkills(x))

  [5] NL:     All students can participate in coding challenges.
       Parsed: ∀x.(Student(x) IMPLIES CanParticipate(x, CodingChallenges))
       Gold:   ∀x (Studen

## 5. Structural Match Summary

A *structural match* checks that the top-level FOL shape (quantifier type, operator) matches the gold. This is approximate — the gold uses short identifiers while the parser uses full names.

In [10]:
import re

def top_shape(fol: str) -> str:
    """Return coarse structural label from a FOL string."""
    fol = fol.strip()
    if fol.startswith("∀"):
        inner = fol.split(".", 1)[-1].strip()
        if "IMPLIES" in inner or "→" in inner or "⇒" in inner:
            return "FORALL-IMPLIES"
        if "AND" in inner or "∧" in inner:
            return "FORALL-AND"
        if "NOT" in inner or "¬" in inner:
            return "FORALL-NOT"
        return "FORALL-ATOMIC"
    if fol.startswith("∃"):
        return "EXISTS-ATOMIC"
    if "IMPLIES" in fol:
        return "IMPLIES"
    if "AND" in fol:
        return "AND"
    if "NOT" in fol or fol.startswith("NOT"):
        return "NOT-ATOMIC"
    return "ATOMIC"

def gold_shape(fol: str) -> str:
    fol = fol.strip()
    if fol.startswith("∀"):
        inner = fol.split(".", 1)[-1].strip()
        if "→" in inner:
            return "FORALL-IMPLIES"
        if "∧" in inner:
            return "FORALL-AND"
        if "¬" in inner:
            return "FORALL-NOT"
        return "FORALL-ATOMIC"
    if fol.startswith("∃"):
        return "EXISTS-ATOMIC"
    return "ATOMIC"

total, matched = 0, 0
mismatches = []

for entry in results:
    if entry["error"]:
        continue
    parsed_list = entry["result"]["premises"]
    gold_list = entry["item"]["premises-FOL"]
    for j, (p, g) in enumerate(zip(parsed_list, gold_list)):
        ps = top_shape(p["fol"])
        gs = gold_shape(g)
        total += 1
        if ps == gs:
            matched += 1
        else:
            mismatches.append({
                "nl": entry["item"]["premises-NL"][j],
                "parsed_fol": p["fol"],
                "gold_fol": g,
                "parsed_shape": ps,
                "gold_shape": gs,
            })

print(f"Structural match: {matched}/{total} = {matched/total:.1%}")
print(f"\nMismatches ({len(mismatches)}):")
for m in mismatches:
    print(f"  NL:     {m['nl']}")
    print(f"  Parsed: {m['parsed_fol']}  [{m['parsed_shape']}]")
    print(f"  Gold:   {m['gold_fol']}  [{m['gold_shape']}]")
    print()

Structural match: 53/112 = 47.3%

Mismatches (59):
  NL:     SQL queries are used to interact with databases and perform operations such as retrieving, inserting, updating, and deleting data.
  Parsed: ((((Used(SQLQueries, Databases) AND Used(SQLQueries, Data)) AND Used(SQLQueries, Data)) AND Update(SQLQueries, Data)) AND Delete(SQLQueries, Data))  [AND]
  Gold:   ForAll(x, SQLQuery(x) → (RetrieveData(x) ∨ InsertData(x) ∨ UpdateData(x) ∨ DeleteData(x)))  [ATOMIC]

  NL:     There exists at least one transportation system that is cost-effective.
  Parsed: ∃x.(TransportationSystem(x) AND CostEffective(x))  [EXISTS-ATOMIC]
  Gold:   Exists(x, Safe(x))  [ATOMIC]

  NL:     If a transportation system is not equipped with modern technology, then it is not cost-effective.
  Parsed: ∀x.((TransportationSystem(x) AND NOT(Equipped(x, ModernTechnology))) IMPLIES NOT(CostEffective(x)))  [FORALL-IMPLIES]
  Gold:   ForAll(x, EquippedWithModernTechnology(x))  [ATOMIC]

  NL:     If a transportation sy

## 6. Predicate Schema and Canonicalization

Shows per-problem predicate schemas and any renames detected.

In [11]:
# NOTE: /premises response does not include schema/renames directly.
# Call /premises and also display predicate names extracted from the AST.

def collect_predicates(ast: dict) -> list[tuple[str, int]]:
    """Walk an AST dict and collect (predicate_name, arity) pairs."""
    results = []
    if ast["type"] == "atomic":
        results.append((ast["predicate"]["name"], len(ast["arguments"])))
    elif ast["type"] == "quantified":
        results.extend(collect_predicates(ast["body"]))
        if ast.get("restrictor"):
            results.extend(collect_predicates(ast["restrictor"]))
    elif ast["type"] == "logical":
        results.extend(collect_predicates(ast["left"]))
        if ast.get("right"):
            results.extend(collect_predicates(ast["right"]))
    return results

display(Markdown("### Predicate vocab per problem"))
for i, entry in enumerate(results):
    if entry["error"]:
        continue
    preds = {}
    for p in entry["result"]["premises"]:
        for name, arity in collect_predicates(p["ast"]):
            preds[f"{name}/{arity}"] = preds.get(f"{name}/{arity}", 0) + 1
    print(f"Problem {i+1}: {dict(sorted(preds.items()))}")

### Predicate vocab per problem

Problem 1: {'Combine/4': 1, 'ConsistsOf/2': 1, 'Delete/2': 1, 'EnsuresIntegrity/2': 1, 'Identifies/3': 1, 'Improve/2': 1, 'Remove/2': 1, 'Standard/1': 1, 'Update/2': 1, 'Used/1': 1, 'Used/2': 3, 'Used/3': 2, 'Used/5': 2, 'UsedToSort/4': 1}
Problem 2: {'CostEffective/0': 1, 'CostEffective/1': 4, 'EcoFriendly/1': 4, 'Equipped/2': 4, 'EquippedWith/1': 1, 'Exist/1': 1, 'Reliable/1': 1, 'Safe/1': 4, 'Safe/2': 1, 'System/2': 1, 'ThoroughlyTested/1': 2, 'TransportationSystem/1': 8, 'TransportationSystem/2': 1}
Problem 3: {'Completed/2': 2, 'Eligible/1': 1, 'Eligible/2': 1, 'HasStatus/2': 2, 'Obtain/2': 1, 'ObtainedAdvisorApproval/1': 1, 'Student/1': 2}
Problem 4: {'Accumulated/2': 1, 'Accumulated/3': 2, 'ApplyingFor/2': 1, 'Approved/2': 1, 'Consider/1': 1, 'CountedToward/2': 1, 'Eligible/2': 1, 'Has/2': 1, 'HasGPA/2': 2, 'HasSummerPriority/2': 1, 'InTrainingProgram/1': 1, 'Includes/2': 1, 'Internship/1': 1, 'InternshipApplication/1': 1, 'Junior/1': 1, 'Missed/2': 1, 'Offered/1': 1, 'Offered/2

## 7. Export Raw Results

In [12]:
output_path = Path("artifacts/reports/b04_premise_parser_eval.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

export = [
    {
        "gold_fol": entry["item"]["premises-FOL"],
        "nl": entry["item"]["premises-NL"],
        "parsed": entry["result"]["premises"] if entry["result"] else None,
        "error": entry["error"],
    }
    for entry in results
]
output_path.write_text(json.dumps(export, indent=2, ensure_ascii=False))
print(output_path.resolve())

/home/phuckhang/MyWorkspace/Exact2026/src/exact/baselines/artifacts/reports/b04_premise_parser_eval.json
